<a href="https://colab.research.google.com/github/divya910agarwal/option-pricing-nifty/blob/main/option_pricing_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np

# ── Load & Parse NSE Option Chain CSV ─────────────────────
def load_nse_csv(filepath):
    # Skip the first 2 header rows, assign manual column names
    call_cols = ['CE_OI', 'CE_CHNG_OI', 'CE_VOLUME', 'CE_IV', 'CE_LTP', 'CE_CHNG',
                 'CE_BID_QTY', 'CE_BID', 'CE_ASK', 'CE_ASK_QTY']
    put_cols  = ['PE_BID_QTY', 'PE_BID', 'PE_ASK', 'PE_ASK_QTY', 'PE_CHNG', 'PE_LTP',
                 'PE_IV', 'PE_VOLUME', 'PE_CHNG_OI', 'PE_OI']

    col_names = ['_dummy'] + call_cols + ['STRIKE'] + put_cols + ['_dummy2']

    df_raw = pd.read_csv(filepath, skiprows=2, header=None, names=col_names)
    df_raw = df_raw.drop(columns=['_dummy', '_dummy2'])

    # Clean: remove commas and convert to numeric
    for col in df_raw.columns:
        df_raw[col] = df_raw[col].astype(str).str.replace(',', '').str.strip()
        df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')

    df_raw = df_raw.dropna(subset=['STRIKE'])
    df_raw = df_raw.reset_index(drop=True)

    # Get spot price from ATM strike area (CE_LTP + PE_LTP ≈ 2x intrinsic near ATM)
    # Use the strike where CE_IV is most populated as a proxy
    spot = df_raw.loc[df_raw['CE_IV'].notna(), 'STRIKE'].median()

    # Reshape into long format: one row per (strike, optionType)
    calls = df_raw[['STRIKE', 'CE_LTP', 'CE_IV', 'CE_OI', 'CE_VOLUME', 'CE_BID', 'CE_ASK']].copy()
    calls.columns = ['strikePrice', 'lastPrice', 'IV', 'OI', 'volume', 'bid', 'ask']
    calls['optionType'] = 'CE'

    puts = df_raw[['STRIKE', 'PE_LTP', 'PE_IV', 'PE_OI', 'PE_VOLUME', 'PE_BID', 'PE_ASK']].copy()
    puts.columns = ['strikePrice', 'lastPrice', 'IV', 'OI', 'volume', 'bid', 'ask']
    puts['optionType'] = 'PE'

    df = pd.concat([calls, puts], ignore_index=True)
    df = df.dropna(subset=['lastPrice', 'strikePrice'])
    df = df[df['lastPrice'] > 0]
    df['expiryDate'] = '30-Mar-2026'

    return df, df_raw

file_path='/content/drive/MyDrive/option-chain-ED-NIFTY-30-Mar-2026.csv'
df, df_raw = load_nse_csv(file_path)

# Estimate spot from put-call parity at ATM
# Spot ≈ strike where |CE_LTP - PE_LTP| is minimum
df_raw_clean = df_raw.dropna(subset=['CE_LTP', 'PE_LTP'])
atm_row = df_raw_clean.iloc[(df_raw_clean['CE_LTP'] - df_raw_clean['PE_LTP']).abs().argsort()[:1]]
S = float(atm_row['STRIKE'].values[0])

print(f" Parsed {len(df)} option records ({len(df[df.optionType=='CE'])} calls, {len(df[df.optionType=='PE'])} puts)")
print(f" Estimated Spot (ATM proxy): {S}")
print(f"Strike range: {df.strikePrice.min()} – {df.strikePrice.max()}")
print(df.head(10))

In [ ]:
import yfinance as yf
import numpy as np

nifty = yf.download("^NSEI", period="1y", auto_adjust=True)['Close']
returns = np.log(nifty / nifty.shift(1)).dropna()
sigma_hist = float(returns.std().iloc[0] * np.sqrt(252))

print(f"Nifty 1Y realised vol : {sigma_hist:.4f}  ({sigma_hist*100:.2f}%)")
print(f"Based on {len(returns)} daily returns")

In [ ]:
import numpy as np
from scipy.stats import norm
from scipy.optimize import brentq
from datetime import datetime

def bs_price(S, K, T, r, sigma, option_type='CE'):
    """Black-Scholes price for European call (CE) or put (PE)."""
    if T <= 0 or sigma <= 0:
        return np.nan
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if option_type == 'CE':
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    else:
        return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

def bs_greeks(S, K, T, r, sigma, option_type='CE'):
    """Delta, Gamma, Theta (per day), Vega (per 1% vol move)."""
    if T <= 0 or sigma <= 0:
        return {'delta': np.nan, 'gamma': np.nan, 'theta': np.nan, 'vega': np.nan}
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    delta = norm.cdf(d1) if option_type == 'CE' else -norm.cdf(-d1)
    gamma = norm.pdf(d1) / (S * sigma * np.sqrt(T))
    theta = (-S * norm.pdf(d1) * sigma / (2 * np.sqrt(T))
             - r * K * np.exp(-r * T) * (norm.cdf(d2) if option_type == 'CE' else norm.cdf(-d2))) / 365
    vega  = S * norm.pdf(d1) * np.sqrt(T) / 100
    return {'delta': delta, 'gamma': gamma, 'theta': theta, 'vega': vega}

def implied_vol(market_price, S, K, T, r, option_type='CE'):
    """Back out implied volatility using Brent's method."""
    try:
        intrinsic = max(S - K, 0) if option_type == 'CE' else max(K - S, 0)
        if market_price <= intrinsic:
            return np.nan
        return brentq(
            lambda s: bs_price(S, K, T, r, s, option_type) - market_price,
            1e-4, 10.0, xtol=1e-6
        )
    except Exception:
        return np.nan

def binomial_tree(S, K, T, r, sigma, N=200, option_type='CE'):
    """CRR binomial tree for European option pricing."""
    dt = T / N
    u  = np.exp(sigma * np.sqrt(dt))
    d  = 1 / u
    p  = (np.exp(r * dt) - d) / (u - d)
    discount = np.exp(-r * dt)
    j  = np.arange(N + 1)
    ST = S * (u ** (N - j)) * (d ** j)
    V  = np.maximum(ST - K, 0) if option_type == 'CE' else np.maximum(K - ST, 0)
    for _ in range(N):
        V = discount * (p * V[:-1] + (1 - p) * V[1:])
    return V[0]
def mc_european_vanilla(S0, K, T, r, sigma, M=100_000,
                      option_type='CE',
                      use_antithetic=True,
                      use_control_variate=True,
                      seed=42):
  """
  Monte Carlo pricer for European options with optional variance reduction.
  Returns (price, standard_error, paths_used).
  Uses single-step GBM since only S_T is needed for European payoff.
  """
  rng = np.random.default_rng(seed)

  # Generate terminal stock prices
  if use_antithetic:
      m_half = M // 2
      z_half = rng.standard_normal(m_half)
      z = np.concatenate([z_half, -z_half])
  else:
      z = rng.standard_normal(M)

  ST = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * z)

  # Payoffs
  if option_type == 'CE':
      payoff = np.maximum(ST - K, 0)
  else:
      payoff = np.maximum(K - ST, 0)

  discount = np.exp(-r * T)
  discounted_payoff = discount * payoff

  # Control variate: use S_T with known E[S_T] = S0 * exp(rT)
  if use_control_variate:
      EST = S0 * np.exp(r * T)
      cov_matrix = np.cov(discounted_payoff, ST)
      c_star = cov_matrix[0, 1] / cov_matrix[1, 1]
      adjusted = discounted_payoff - c_star * (ST - EST)
  else:
      adjusted = discounted_payoff

  price   = adjusted.mean()
  std_err = adjusted.std(ddof=1) / np.sqrt(len(adjusted))
  return price, std_err, len(adjusted)


S      = 24550.0
r      = 0.065
expiry = datetime(2026, 3, 30)
today  = datetime.today()
T      = (expiry - today).days / 365

print(f"Days to expiry : {(expiry - today).days}")
print(f"T (years)      : {T:.4f}")
print(f"Spot           : {S}")
print(f"Sigma hist     : {sigma_hist:.4f}")

df_priced = df[df['lastPrice'] > 0].copy()

bs_prices, iv_list = [], []
delta_list, gamma_list, theta_list, vega_list = [], [], [], []

for i in range(len(df_priced)):
    row = df_priced.iloc[i]
    bs_prices.append(bs_price(S, row['strikePrice'], T, r, sigma_hist, row['optionType']))
    iv_list.append(implied_vol(row['lastPrice'], S, row['strikePrice'], T, r, row['optionType']))
    g = bs_greeks(S, row['strikePrice'], T, r, sigma_hist, row['optionType'])
    delta_list.append(g['delta'])
    gamma_list.append(g['gamma'])
    theta_list.append(g['theta'])
    vega_list.append(g['vega'])

df_priced['bs_price']     = bs_prices
df_priced['IV_calc']      = iv_list
df_priced['delta']        = delta_list
df_priced['gamma']        = gamma_list
df_priced['theta']        = theta_list
df_priced['vega']         = vega_list
df_priced['mispricing']   = df_priced['lastPrice'] - df_priced['bs_price']
df_priced['misprice_pct'] = df_priced['mispricing'] / df_priced['bs_price'] * 100

cols_show = ['strikePrice', 'optionType', 'lastPrice', 'bs_price',
             'IV_calc', 'mispricing', 'misprice_pct', 'delta', 'gamma', 'theta', 'vega']
print(df_priced[cols_show].dropna(subset=['bs_price']).head(15).to_string(index=False))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

# keep only liquid options for signal analysis
df_liquid = df_priced[
    (df_priced['volume'] > 10) | (df_priced['OI'] > 100)
].copy()
df_liquid = df_liquid.dropna(subset=['bs_price', 'IV_calc'])
df_liquid = df_liquid[df_liquid['IV_calc'] < 2.0]
df_liquid = df_liquid[df_liquid['misprice_pct'].abs() < 50]

# separate calls and puts
calls = df_liquid[df_liquid['optionType'] == 'CE'].sort_values('strikePrice').copy()
puts  = df_liquid[df_liquid['optionType'] == 'PE'].sort_values('strikePrice').copy()

# fit quadratic through the IV points -> this IS the volatility smile
# the smile captures how market prices vol differently across strikes
# (higher IV for OTM puts = put skew, for OTM calls = call smile)
def fit_smile(df_side):
    valid = df_side.dropna(subset=['IV_calc'])
    return np.polyfit(valid['strikePrice'], valid['IV_calc'], deg=2)

call_coeffs = fit_smile(calls)
put_coeffs  = fit_smile(puts)

calls['IV_smile_fit'] = np.polyval(call_coeffs, calls['strikePrice'])
puts['IV_smile_fit']  = np.polyval(put_coeffs,  puts['strikePrice'])

# mispricing = deviation of each option's IV from the smile
# if IV_calc >> smile fit -> option is expensive relative to its peers
calls['smile_spread'] = calls['IV_calc'] - calls['IV_smile_fit']
puts['smile_spread']  = puts['IV_calc']  - puts['IV_smile_fit']

threshold = 0.015  # 1.5pp deviation

call_signals, put_signals = [], []
for x in calls['smile_spread']:
    if x > threshold:    call_signals.append('SELL')
    elif x < -threshold: call_signals.append('BUY')
    else:                call_signals.append('NEUTRAL')

for x in puts['smile_spread']:
    if x > threshold:    put_signals.append('SELL')
    elif x < -threshold: put_signals.append('BUY')
    else:                put_signals.append('NEUTRAL')

calls['rel_signal'] = call_signals
puts['rel_signal']  = put_signals

print(f"Realised vol (sigma_hist): {sigma_hist:.4f} ({sigma_hist*100:.2f}%)\n")

for name, side in [('CALLS', calls), ('PUTS', puts)]:
    print(f"--- {name}: IV vs Smile ---")
    print(side[['strikePrice', 'lastPrice', 'IV_calc', 'IV_smile_fit',
                'smile_spread', 'rel_signal']].to_string(index=False))
    print()

In [ ]:
bt_prices = []
for i in range(len(df_liquid)):
    row = df_liquid.iloc[i]
    bt_prices.append(binomial_tree(S, row['strikePrice'], T, r, sigma_hist, option_type=row['optionType']))

df_liquid['bt_price']   = bt_prices
df_liquid['bs_bt_diff'] = df_liquid['bs_price'] - df_liquid['bt_price']

print("BS vs Binomial Tree (should be near zero):")
print(df_liquid[['strikePrice', 'optionType', 'bs_price', 'bt_price', 'bs_bt_diff']].to_string(index=False))

In [ ]:
fig = plt.figure(figsize=(16, 12))
gs  = gridspec.GridSpec(2, 2, hspace=0.35, wspace=0.3)
ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[1, 0])
ax4 = fig.add_subplot(gs[1, 1])

def scatter_smile(ax, side, name, coeffs):
    neutral  = side[side['rel_signal'] == 'NEUTRAL']
    sell_pts = side[side['rel_signal'] == 'SELL']
    buy_pts  = side[side['rel_signal'] == 'BUY']
    ax.scatter(neutral['strikePrice'],  neutral['IV_calc'],  color='steelblue', s=40, label='Neutral',     zorder=3)
    ax.scatter(sell_pts['strikePrice'], sell_pts['IV_calc'], color='crimson',   s=60, label='Sell (rich)', zorder=4)
    ax.scatter(buy_pts['strikePrice'],  buy_pts['IV_calc'],  color='green',     s=60, label='Buy (cheap)', zorder=4)
    x_range = np.linspace(side['strikePrice'].min(), side['strikePrice'].max(), 300)
    ax.plot(x_range, np.polyval(coeffs, x_range), 'k--', linewidth=1.2, label='Smile fit')
    ax.axvline(S, color='gray', linestyle=':', linewidth=1, label=f'Spot ({int(S)})')
    for _, row in pd.concat([sell_pts, buy_pts]).iterrows():
        ax.annotate(f"{int(row['strikePrice'])}", (row['strikePrice'], row['IV_calc']),
                    textcoords="offset points", xytext=(5, 5), fontsize=8)
    ax.set_title(f'IV Smile — {name}', fontweight='bold')
    ax.set_xlabel('Strike Price')
    ax.set_ylabel('Implied Volatility')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y*100:.1f}%'))
    ax.legend(fontsize=8)

scatter_smile(ax1, calls, 'Calls', call_coeffs)
scatter_smile(ax2, puts,  'Puts',  put_coeffs)

for ax, side, name in [(ax3, calls, 'Calls'), (ax4, puts, 'Puts')]:
    colors = ['crimson' if x > 0.015 else ('green' if x < -0.015 else 'steelblue')
              for x in side['smile_spread']]
    ax.bar(side['strikePrice'], side['smile_spread'] * 100, color=colors, width=40)
    ax.axhline(0,    color='black',   linewidth=0.8)
    ax.axhline(1.5,  color='crimson', linewidth=0.8, linestyle='--', label='+1.5% threshold')
    ax.axhline(-1.5, color='green',   linewidth=0.8, linestyle='--', label='-1.5% threshold')
    ax.axvline(S, color='gray', linestyle=':', linewidth=1)
    ax.set_title(f'IV Deviation from Smile — {name}', fontweight='bold')
    ax.set_xlabel('Strike Price')
    ax.set_ylabel('IV Spread (pp)')
    ax.legend(fontsize=8)

fig.suptitle(f'NIFTY Option Chain  |  Expiry: 30-Mar-2026  |  Spot: {S}  |  Realised Vol: {sigma_hist*100:.1f}%',
             fontsize=12, fontweight='bold')
plt.savefig('nifty_option_analysis.png', dpi=150, bbox_inches='tight')
plt.show()